# DEEP R + W2 growth on multi-MNIST 2-task with LTU hidden layer

Same architecture as `deep_r_2layer.ipynb` (1 hidden layer of LTUs, multi-MNIST
2-task, per-hidden-unit one-hot init for W2). Two changes:

1. **W2 also prunes** on sign flip — but only for hidden units whose age has
   reached `age_threshold`.
2. **W2 grows new connections** for the same eligible hidden units. Whenever
   the total number of active W2 entries falls below `output_budget`, up to
   `max_gen_per_step` new W2 entries are sampled per step from the pool of
   `(eligible hidden, any output)` slots that are currently inactive. New
   weights have magnitude `output_init_magnitude` with random sign — same
   convention as the original DEEP R rewiring (Bellec et al.).

W1 is unchanged from `deep_r_2layer.ipynb` (only prunes, never grows). The age
counter is per-hidden-unit and increments every step; with `age_threshold=100k`
the default behavior matches `deep_r_2layer.ipynb` for the first 100k steps,
then opens up W2 dynamics.

## Setup

In [21]:
import os
import sys

REPO_ROOT = '/home/edan/local_projects/phd_research'
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'phd', 'structure_search')):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from phd.jax_core.models import ltu

# Multi-MNIST 2-task layout
N_TASKS = 2
NUM_CLASSES = 10
INPUT_PER_TASK = 784
INPUT_DIM = INPUT_PER_TASK * N_TASKS              # 1568
OUTPUT_DIM = NUM_CLASSES * N_TASKS                # 20

# 2-layer LTU layout — each hidden unit hardwired to one output, equally split.
N_HIDDEN = 100                                    # must divide OUTPUT_DIM
HIDDEN_PER_OUTPUT = N_HIDDEN // OUTPUT_DIM        # 5
INPUT_FANIN = 256                                 # initial active input connections per hidden unit

assert N_HIDDEN % OUTPUT_DIM == 0, "N_HIDDEN must be divisible by OUTPUT_DIM"

print('JAX device:', jax.devices()[0])
print(f'INPUT_DIM={INPUT_DIM}  N_HIDDEN={N_HIDDEN}  OUTPUT_DIM={OUTPUT_DIM}'
      f'  HIDDEN_PER_OUTPUT={HIDDEN_PER_OUTPUT}  INPUT_FANIN={INPUT_FANIN}')

JAX device: cuda:0
INPUT_DIM=1568  N_HIDDEN=100  OUTPUT_DIM=20  HIDDEN_PER_OUTPUT=5  INPUT_FANIN=256


## Data

In [22]:
def load_data():
    """MNIST standardized per-pixel."""
    from data import load_dataset
    images, labels, _, _ = load_dataset('mnist', split='train')
    images = np.asarray(images, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)
    mean = images.mean(axis=0, keepdims=True)
    std = images.std(axis=0, keepdims=True)
    normalized = (images - mean) / np.maximum(std, 1e-3)
    return jnp.asarray(normalized), jnp.asarray(labels)


images, labels = load_data()
print('images:', images.shape, '   labels:', labels.shape)

images: (60000, 784)    labels: (60000,)


## Architecture

Forward: `z1 = x @ (W1 * M1); h = ltu(z1); logits = h @ (W2 * M2)`. Per-task
softmax CE summed over 2 tasks. No biases.

In [23]:
def forward(W1, M1, W2, M2, x):
    z1 = x @ (W1 * M1)                              # (N_HIDDEN,)
    h = ltu(z1)                                     # binary {0,1} forward, sigmoid-STE backward
    logits = h @ (W2 * M2)                          # (OUTPUT_DIM,)
    return logits, h, z1


def loss_fn(W1, M1, W2, M2, x, y):
    logits, _, _ = forward(W1, M1, W2, M2, x)
    logits_pt = logits.reshape(N_TASKS, NUM_CLASSES)
    lp = jax.nn.log_softmax(logits_pt, axis=-1)
    return -jnp.mean(jnp.sum(jax.nn.one_hot(y, NUM_CLASSES) * lp, axis=-1))


def make_sample(images, labels, key):
    k1, k2 = jax.random.split(key)
    idx1 = jax.random.randint(k1, (), 0, images.shape[0])
    idx2 = jax.random.randint(k2, (), 0, images.shape[0])
    x = jnp.concatenate([images[idx1], images[idx2]])
    y = jnp.array([labels[idx1], labels[idx2]])
    return x, y

## Init

Per hidden unit: pick `INPUT_FANIN` random input pixels; one-hot to its assigned
output. Weights at active entries are Kaiming-uniform with the appropriate fan-in.
Future extensions (more outgoing connections, feature growth) just modify these
masks before passing them to the train functions.

In [24]:
def init_2layer_ltu(seed=0, n_hidden=N_HIDDEN, input_fanin=INPUT_FANIN):
    """Init W1, M1, W2, M2.

    M1: each column has `input_fanin` ones at random rows.
    M2: one-hot row, hidden unit i routes to output i // (n_hidden // OUTPUT_DIM).
    """
    k = jax.random.key(seed)
    k_m1, k_w1, k_w2 = jax.random.split(k, 3)

    # M1: per hidden unit, sample input_fanin random inputs.
    keys = jax.random.split(k_m1, n_hidden)

    def per_unit(key):
        noise = jax.random.uniform(key, (INPUT_DIM,))
        idx = jnp.argsort(-noise)[:input_fanin]
        return jnp.zeros(INPUT_DIM, dtype=jnp.int32).at[idx].set(1)

    M1_T = jax.vmap(per_unit)(keys)                              # (N_HIDDEN, INPUT_DIM)
    M1 = M1_T.T                                                  # (INPUT_DIM, N_HIDDEN)

    w1_bound = jnp.sqrt(3.0 / float(input_fanin))
    W1 = jax.random.uniform(k_w1, (INPUT_DIM, n_hidden),
                            minval=-w1_bound, maxval=w1_bound) * M1

    hidden_per_output = n_hidden // OUTPUT_DIM
    h2o = jnp.arange(n_hidden) // hidden_per_output              # (N_HIDDEN,)
    M2 = jax.nn.one_hot(h2o, OUTPUT_DIM, dtype=jnp.int32)        # (N_HIDDEN, OUTPUT_DIM)
    w2_bound = jnp.sqrt(3.0 / float(hidden_per_output))
    W2 = jax.random.uniform(k_w2, (n_hidden, OUTPUT_DIM),
                            minval=-w2_bound, maxval=w2_bound) * M2
    return W1, M1, W2, M2


# Sanity-check the init.
_W1, _M1, _W2, _M2 = init_2layer_ltu(seed=0)
print(f'M1 active per hidden unit: min={int(_M1.sum(0).min())} '
      f'mean={float(_M1.sum(0).mean())} max={int(_M1.sum(0).max())}')
print(f'M2 active per hidden unit: {int(_M2.sum(1).min())} (should be 1)')
print(f'M2 active per output: {int(_M2.sum(0).min())}/{int(_M2.sum(0).max())} '
      f'(should both be {HIDDEN_PER_OUTPUT})')

M1 active per hidden unit: min=256 mean=256.0 max=256
M2 active per hidden unit: 1 (should be 1)
M2 active per output: 5/5 (should both be 5)


## DEEP R training (W1 prunes; W2 prunes + grows past age threshold)

Per step:

  W_new = W − lr·grad − lr·l1·sign(W) + sqrt(2·lr·T)·noise

apply current mask, then:

- **W1**: deactivate any entry whose sign just flipped (always on).
- **W2**: deactivate sign-flipped entries **only for hidden units with
  `age >= age_threshold`**. Then if `M2.sum() < output_budget`, sample up to
  `max_gen_per_step` new entries from the eligible pool (eligible hidden unit
  × inactive output slot), set them to `±output_init_magnitude` with random
  sign.

In [25]:
def train_deep_r(W1_init, M1_init, W2_init, M2_init, images, labels, *,
                 lr=2**-7,
                 l1=1e-4,
                 temperature=1e-7,
                 n_steps=500_000,
                 snapshot_every=2_000,
                 permute_period=0,
                 age_threshold=100_000,
                 max_gen_age=None,
                 output_budget=OUTPUT_DIM * N_HIDDEN // 2,
                 output_init_magnitude=1e-3,
                 max_gen_per_step=1,
                 seed=0):
    """DEEP R on W1 (deactivate-only) and W2 (deactivate + grow), with both
    W2 dynamics gated by a per-hidden-unit age threshold.

    `age_threshold`: age at which a unit's W2 first prunes/grows.
    `max_gen_age`: optional upper bound. When set, growth stops for units with
        `age >= max_gen_age`; pruning still applies.

    Carry includes scalar `cum_deact_W2_within` and `cum_deact_W2_cross` so we
    can report total cumulative W2 prunings split by within/cross task."""
    n_chunks = n_steps // snapshot_every
    perm0_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    age_init = jnp.zeros(N_HIDDEN, dtype=jnp.int32)

    # Within/cross masks for W2, available inside the scan.
    hidden_task_j = (jnp.arange(N_HIDDEN) // HIDDEN_PER_OUTPUT) // NUM_CLASSES
    output_task_j = jnp.arange(OUTPUT_DIM) // NUM_CLASSES
    same_task_ho_j = (hidden_task_j[:, None] == output_task_j[None, :])     # (HIDDEN, OUT)

    init_carry = (W1_init, M1_init.astype(jnp.int32),
                  W2_init, M2_init.astype(jnp.int32),
                  perm0_init, perm1_init,
                  age_init,
                  jnp.array(0, dtype=jnp.int32),                            # t
                  jnp.array(0, dtype=jnp.int32),                            # cum W2 deact within
                  jnp.array(0, dtype=jnp.int32))                            # cum W2 deact cross
    noise_scale = jnp.sqrt(2.0 * lr * temperature)

    def step_fn(carry, key):
        (W1, M1, W2, M2, perm0, perm1, age, t,
         cum_dw, cum_dc) = carry
        (data_key, n1_key, n2_key, perm_key,
         gen_pos_key, gen_sign_key) = jax.random.split(key, 6)
        x, y_raw = make_sample(images, labels, data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])

        def _loss(W1_, W2_):
            return loss_fn(W1_, M1, W2_, M2, x, y)
        loss, (g1, g2) = jax.value_and_grad(_loss, argnums=(0, 1))(W1, W2)

        s1 = jnp.sign(W1)
        s2 = jnp.sign(W2)
        n1 = jax.random.normal(n1_key, W1.shape) * noise_scale
        n2 = jax.random.normal(n2_key, W2.shape) * noise_scale

        W1_new = (W1 - lr * g1 - lr * l1 * s1 + n1) * M1
        W2_new = (W2 - lr * g2 - lr * l1 * s2 + n2) * M2

        deact1 = (jnp.sign(W1_new) != s1) & (M1 == 1)
        M1_new = M1 * (1 - deact1.astype(jnp.int32))
        W1_new = W1_new * M1_new

        eligible_h = (age >= age_threshold)
        deact2 = ((jnp.sign(W2_new) != s2) & (M2 == 1)
                  & eligible_h[:, None])
        M2_new = M2 * (1 - deact2.astype(jnp.int32))
        W2_new = W2_new * M2_new

        # Cumulative W2 deactivation tally split by within/cross task.
        d2_int = deact2.astype(jnp.int32)
        cum_dw_new = cum_dw + jnp.sum(d2_int * same_task_ho_j.astype(jnp.int32))
        cum_dc_new = cum_dc + jnp.sum(d2_int * (~same_task_ho_j).astype(jnp.int32))

        # W2 growth: gated by [age_threshold, max_gen_age) when max_gen_age set.
        if max_gen_age is None:
            eligible_grow_h = eligible_h
        else:
            eligible_grow_h = eligible_h & (age < max_gen_age)

        n_active2 = M2_new.sum()
        deficit = jnp.maximum(output_budget - n_active2, 0)
        n_to_add = jnp.minimum(max_gen_per_step, deficit).astype(jnp.int32)

        is_eligible_slot = eligible_grow_h[:, None] & (M2_new == 0)
        flat_elig = is_eligible_slot.reshape(-1)
        rand_pos = jax.random.uniform(gen_pos_key, (N_HIDDEN * OUTPUT_DIM,))
        score = jnp.where(flat_elig, rand_pos, -1.0)
        top_scores, top_idx = jax.lax.top_k(score, max_gen_per_step)
        positions = jnp.arange(max_gen_per_step)
        will_activate = (positions < n_to_add) & (top_scores > -0.5)

        signs = jax.random.choice(gen_sign_key, jnp.array([-1.0, 1.0]),
                                   shape=(max_gen_per_step,))
        new_vals = signs * output_init_magnitude

        flat_M2 = M2_new.reshape(-1)
        flat_W2 = W2_new.reshape(-1)
        flat_M2 = flat_M2.at[top_idx].set(
            jnp.where(will_activate, 1, flat_M2[top_idx]))
        flat_W2 = flat_W2.at[top_idx].set(
            jnp.where(will_activate, new_vals, flat_W2[top_idx]))
        M2_new = flat_M2.reshape(M2_new.shape)
        W2_new = flat_W2.reshape(W2_new.shape)

        age_new = age + 1
        t_next = t + 1
        if permute_period > 0:
            should_perm = (t_next >= permute_period) & (t_next % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0_new = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1_new = jnp.where(should_perm & (which == 1), new_perm, perm1)
        else:
            perm0_new = perm0
            perm1_new = perm1

        return (W1_new, M1_new, W2_new, M2_new,
                perm0_new, perm1_new, age_new, t_next,
                cum_dw_new, cum_dc_new), loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        (W1, M1, W2, M2, _perm0, _perm1, _age, t,
         cum_dw, cum_dc) = carry
        snap = dict(
            step=t,
            avg_loss=losses.mean(),
            M1=M1,
            M2=M2,
            n_active_W1=M1.sum(),
            n_active_W2=M2.sum(),
            cum_deact_W2_within=cum_dw,
            cum_deact_W2_cross=cum_dc,
        )
        return carry, snap

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W1'] = jax.device_get(final_carry[0])
    snaps['final_M1'] = jax.device_get(final_carry[1])
    snaps['final_W2'] = jax.device_get(final_carry[2])
    snaps['final_M2'] = jax.device_get(final_carry[3])
    return snaps


## Baseline (no-prune)

Same init, plain SGD, masks frozen at the initial values. Loss curve is the
no-prune reference.

In [26]:
def train_baseline(W1_init, M1_init, W2_init, M2_init, images, labels, *,
                   lr=2**-7,
                   n_steps=500_000,
                   snapshot_every=2_000,
                   permute_period=0,
                   seed=0):
    """Plain SGD with frozen masks. No L1, no noise, no deactivation.

    `permute_period`: 0 = stationary. >0 = same non-stationary mechanism as
    train_deep_r so the loss curves are directly comparable."""
    n_chunks = n_steps // snapshot_every
    perm0_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    init_carry = (W1_init, W2_init, perm0_init, perm1_init,
                  jnp.array(0, dtype=jnp.int32))

    def step_fn(carry, key):
        W1, W2, perm0, perm1, t = carry
        data_key, perm_key = jax.random.split(key)
        x, y_raw = make_sample(images, labels, data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])
        def _loss(W1_, W2_):
            return loss_fn(W1_, M1_init, W2_, M2_init, x, y)
        loss, (g1, g2) = jax.value_and_grad(_loss, argnums=(0, 1))(W1, W2)
        W1 = (W1 - lr * g1) * M1_init
        W2 = (W2 - lr * g2) * M2_init
        t_next = t + 1
        if permute_period > 0:
            should_perm = (t_next >= permute_period) & (t_next % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0 = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1 = jnp.where(should_perm & (which == 1), new_perm, perm1)
        return (W1, W2, perm0, perm1, t_next), loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        W1, W2, _p0, _p1, t = carry
        return carry, dict(step=t, avg_loss=losses.mean())

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W1'] = jax.device_get(final_carry[0])
    snaps['final_W2'] = jax.device_get(final_carry[1])
    return snaps

## Run

The DEEP R run takes longer because of the noise updates and the per-chunk
mask snapshots. Reduce `n_steps` for fast iteration.

In [27]:
# Shared init (same seed -> same starting topology and weights for both runs).
W1_init, M1_init, W2_init, M2_init = init_2layer_ltu(seed=0)

# DEEP R hyperparameters - edit freely.
DEEP_R_CONFIG = dict(
    lr=2**-7,
    l1=1e-4,
    temperature=1e-7,
    n_steps=500_000,
    snapshot_every=2_000,
    permute_period=0,                         # 0 = stationary
    age_threshold=100_000,                    # steps a unit must live before its W2 prunes/grows
    max_gen_age=300_000,                      # None = no upper bound; int = stop growing at this age
    output_budget=OUTPUT_DIM * N_HIDDEN // 2, # target # of active W2 entries
    output_init_magnitude=1e-3,               # |w| of newly-generated W2 entries
    max_gen_per_step=1,                       # cap on new W2 entries per step
    seed=0,
)

train_deep_r_jit = jax.jit(
    train_deep_r,
    static_argnames=('lr', 'l1', 'temperature', 'n_steps', 'snapshot_every',
                     'permute_period', 'age_threshold', 'max_gen_age',
                     'output_budget', 'output_init_magnitude',
                     'max_gen_per_step', 'seed'),
)
train_baseline_jit = jax.jit(
    train_baseline,
    static_argnames=('lr', 'n_steps', 'snapshot_every', 'permute_period', 'seed'),
)

print('Running DEEP R...')
deep_r_snaps = train_deep_r_jit(W1_init, M1_init, W2_init, M2_init,
                                images, labels, **DEEP_R_CONFIG)
deep_r_snaps['M1_init'] = np.asarray(M1_init)
deep_r_snaps['M2_init'] = np.asarray(M2_init)
print(f'  final loss: {float(deep_r_snaps["avg_loss"][-1]):.4f}')
print(f'  W1 active: {int(deep_r_snaps["n_active_W1"][-1])} / {int(M1_init.sum())}')
print(f'  W2 active: {int(deep_r_snaps["n_active_W2"][-1])} / target {DEEP_R_CONFIG["output_budget"]}'
      f' (init {int(M2_init.sum())}, max possible {N_HIDDEN * OUTPUT_DIM})')

print('\nRunning baseline (no-prune, no-grow)...')
baseline_snaps = train_baseline_jit(W1_init, M1_init, W2_init, M2_init, images, labels,
                                     lr=DEEP_R_CONFIG['lr'],
                                     n_steps=DEEP_R_CONFIG['n_steps'],
                                     snapshot_every=DEEP_R_CONFIG['snapshot_every'],
                                     permute_period=DEEP_R_CONFIG['permute_period'],
                                     seed=DEEP_R_CONFIG['seed'])
print(f'  final loss: {float(baseline_snaps["avg_loss"][-1]):.4f}')


Running DEEP R...
  final loss: 0.3359
  W1 active: 1772 / 25600
  W2 active: 635 / target 1000 (init 100, max possible 2000)

Running baseline (no-prune, no-grow)...
  final loss: 0.3127


## Metrics — cross/within prune accounting

Each hidden unit had a random initial fanin split between within-task and
cross-task input pixels. We track per-unit and average:

- `pruned_within`, `pruned_cross`: how many initial-active connections of each type
  have been deactivated, averaged across hidden units.
- `frac_remaining_within`, `frac_remaining_cross`: fraction of each type's initial-
  active connections that are still active, averaged across hidden units.

In [28]:
# Task assignments (matching common.hidden_unit_task_ids etc.).
INPUT_TASK  = np.arange(INPUT_DIM)  // INPUT_PER_TASK                       # (IN,)
HIDDEN_TASK = (np.arange(N_HIDDEN) // HIDDEN_PER_OUTPUT) // NUM_CLASSES     # (HIDDEN,)
OUTPUT_TASK = np.arange(OUTPUT_DIM) // NUM_CLASSES                          # (OUT,)
SAME_TASK_IH = (INPUT_TASK[:, None] == HIDDEN_TASK[None, :])                # (IN, HIDDEN) bool
SAME_TASK_HO = (HIDDEN_TASK[:, None] == OUTPUT_TASK[None, :])               # (HIDDEN, OUT) bool


def compute_prune_metrics(snaps):
    """From M1_init and the per-snapshot M1 history, compute per-unit cross/within
    fraction-remaining arrays. Shapes: (n_chunks, N_HIDDEN)."""
    M1_init = np.asarray(snaps['M1_init']).astype(np.int32)
    M1s = np.asarray(snaps['M1']).astype(np.int32)              # (n_chunks, IN, HIDDEN)

    init_within = (M1_init * SAME_TASK_IH.astype(M1_init.dtype)).sum(axis=0)   # (HIDDEN,)
    init_cross  = (M1_init * (~SAME_TASK_IH).astype(M1_init.dtype)).sum(axis=0)

    same_h    = SAME_TASK_IH.astype(M1s.dtype)[None, :, :]
    not_same  = (~SAME_TASK_IH).astype(M1s.dtype)[None, :, :]
    active_within = (M1s * same_h).sum(axis=1)                  # (n_chunks, HIDDEN)
    active_cross  = (M1s * not_same).sum(axis=1)

    den_w = np.maximum(init_within[None, :], 1)
    den_c = np.maximum(init_cross[None, :],  1)
    frac_remaining_within = active_within / den_w
    frac_remaining_cross  = active_cross  / den_c

    return dict(
        steps=np.asarray(snaps['step']),
        init_within_per_unit=init_within,
        init_cross_per_unit=init_cross,
        frac_remaining_within=frac_remaining_within,
        frac_remaining_cross=frac_remaining_cross,
    )


def compute_w2_metrics(snaps):
    """Per-snapshot W2 active counts (total / within / cross) and cumulative
    deactivation tallies (within / cross)."""
    M2s = np.asarray(snaps['M2']).astype(np.int32)              # (n_chunks, HIDDEN, OUT)
    same_ho     = SAME_TASK_HO.astype(M2s.dtype)[None, :, :]
    not_same_ho = (~SAME_TASK_HO).astype(M2s.dtype)[None, :, :]
    active_total  = M2s.sum(axis=(1, 2))
    active_within = (M2s * same_ho).sum(axis=(1, 2))
    active_cross  = (M2s * not_same_ho).sum(axis=(1, 2))
    return dict(
        steps=np.asarray(snaps['step']),
        active_total=active_total,
        active_within=active_within,
        active_cross=active_cross,
        cum_deact_within=np.asarray(snaps['cum_deact_W2_within']),
        cum_deact_cross=np.asarray(snaps['cum_deact_W2_cross']),
    )


metrics = compute_prune_metrics(deep_r_snaps)
w2_metrics = compute_w2_metrics(deep_r_snaps)
print(f'mean initial W1 within / cross per unit: '
      f'{metrics["init_within_per_unit"].mean():.1f}'
      f' / {metrics["init_cross_per_unit"].mean():.1f}')
print(f'final W1 fraction remaining within / cross: '
      f'{metrics["frac_remaining_within"][-1].mean():.3f}'
      f' / {metrics["frac_remaining_cross"][-1].mean():.3f}')
print(f'final W2 active total / within / cross: '
      f'{int(w2_metrics["active_total"][-1])}'
      f' / {int(w2_metrics["active_within"][-1])}'
      f' / {int(w2_metrics["active_cross"][-1])}')
print(f'cumulative W2 prunings within / cross: '
      f'{int(w2_metrics["cum_deact_within"][-1])}'
      f' / {int(w2_metrics["cum_deact_cross"][-1])}')


mean initial W1 within / cross per unit: 127.5 / 128.5
final W1 fraction remaining within / cross: 0.139 / 0.000
final W2 active total / within / cross: 635 / 618 / 17
cumulative W2 prunings within / cross: 24282 / 53717


## Plots

In [29]:
WITHIN_COLOR = '#1f77b4'   # blue
CROSS_COLOR  = '#d62728'   # red


def plot_loss(deep_r_snaps, baseline_snaps):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.asarray(deep_r_snaps['step']),
                             y=np.asarray(deep_r_snaps['avg_loss']),
                             mode='lines', name='DEEP R + W2 growth'))
    fig.add_trace(go.Scatter(x=np.asarray(baseline_snaps['step']),
                             y=np.asarray(baseline_snaps['avg_loss']),
                             mode='lines', name='baseline (no prune/grow)',
                             line=dict(dash='dot')))
    fig.update_layout(title='Loss over training',
                      xaxis_title='step', yaxis_title='mean loss over snapshot',
                      width=900, height=420)
    fig.show()
    return fig


def plot_remaining_fractions(metrics):
    steps = metrics['steps']
    rw = metrics['frac_remaining_within'].mean(axis=-1)
    rc = metrics['frac_remaining_cross'].mean(axis=-1)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=steps, y=rw, mode='lines', name='within-task remaining',
                             line=dict(color=WITHIN_COLOR)))
    fig.add_trace(go.Scatter(x=steps, y=rc, mode='lines', name='cross-task remaining',
                             line=dict(color=CROSS_COLOR)))
    fig.update_layout(title='W1: fraction of initial input connections remaining',
                      xaxis_title='step',
                      yaxis_title='fraction remaining',
                      yaxis=dict(range=[0, 1.05]),
                      width=900, height=420)
    fig.show()
    return fig


def plot_w2_active_per_unit(w2_metrics, output_budget):
    """Average # of active W2 within/cross connections per hidden unit."""
    steps = w2_metrics['steps']
    aw = w2_metrics['active_within'] / N_HIDDEN
    ac = w2_metrics['active_cross']  / N_HIDDEN
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=steps, y=aw, mode='lines', name='within-task active',
                             line=dict(color=WITHIN_COLOR)))
    fig.add_trace(go.Scatter(x=steps, y=ac, mode='lines', name='cross-task active',
                             line=dict(color=CROSS_COLOR)))
    fig.add_hline(y=output_budget / N_HIDDEN, line_dash='dot',
                  annotation_text=f'budget / N_HIDDEN = {output_budget / N_HIDDEN:.1f}')
    fig.update_layout(title='W2: active output connections per hidden unit',
                      xaxis_title='step',
                      yaxis_title='# active per hidden unit',
                      width=900, height=420)
    fig.show()
    return fig


def plot_w2_cumulative_pruned(w2_metrics):
    """Cumulative W2 within/cross deactivations over training."""
    steps = w2_metrics['steps']
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=steps, y=w2_metrics['cum_deact_within'],
                             mode='lines', name='within-task pruned (cumulative)',
                             line=dict(color=WITHIN_COLOR)))
    fig.add_trace(go.Scatter(x=steps, y=w2_metrics['cum_deact_cross'],
                             mode='lines', name='cross-task pruned (cumulative)',
                             line=dict(color=CROSS_COLOR)))
    fig.update_layout(title='W2: cumulative output connections pruned',
                      xaxis_title='step',
                      yaxis_title='# cumulative deactivations',
                      width=900, height=420)
    fig.show()
    return fig


In [30]:
plot_loss(deep_r_snaps, baseline_snaps)
plot_remaining_fractions(metrics)
plot_w2_active_per_unit(w2_metrics, DEEP_R_CONFIG['output_budget'])
plot_w2_cumulative_pruned(w2_metrics);